In [ ]:
from pyspark.sql.functions import (
    col,
    avg,
    stddev,
    max,
    min,
    sum,
    count,
    when
)

from pyspark.sql.window import Window

In [ ]:
# ============================================================
# 1. Lecture du dataset temporal_features
# ============================================================

df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/temporal_features/"
)

In [ ]:
# ============================================================
# 2. Vérification légère
# ============================================================

df.printSchema()

df.select(
    "LCLid",
    "tstp",
    "energy_kwh",
    "lag_1h",
    "lag_24h",
    "lag_7d",
    "target_energy_next_30min"
).show(5, False)

In [ ]:
# ============================================================
# 3. Test sur un échantillon
# ============================================================

df_sample = df.limit(100000)

In [ ]:
# ============================================================
# 4. Rolling Features sur 24h
# ============================================================

window_spec = Window.partitionBy("LCLid").orderBy("tstp")

rolling_24h = window_spec.rowsBetween(-48, 0)

df_sample = df_sample.withColumn(
    "rolling_mean_24h",
    avg("energy_kwh").over(rolling_24h)
).withColumn(
    "rolling_std_24h",
    stddev("energy_kwh").over(rolling_24h)
).withColumn(
    "rolling_max_24h",
    max("energy_kwh").over(rolling_24h)
).withColumn(
    "rolling_min_24h",
    min("energy_kwh").over(rolling_24h)
)

In [ ]:
# ============================================================
# 5. Vérification des Rolling Features
# ============================================================

df_sample.select(
    "LCLid",
    "tstp",
    "energy_kwh",
    "rolling_mean_24h",
    "rolling_std_24h",
    "rolling_max_24h",
    "rolling_min_24h"
).show(10, False)

In [ ]:
# ============================================================
# 6. Agrégations hebdomadaires
# ============================================================

weekly_df = df_sample.groupBy(
    "LCLid",
    "year",
    "week_of_year"
).agg(
    avg("energy_kwh").alias("weekly_avg_energy"),
    max("energy_kwh").alias("weekly_peak_energy"),
    stddev("energy_kwh").alias("weekly_std_energy")
)

In [ ]:
# ============================================================
# 7. Jointure des agrégations hebdomadaires
# ============================================================

df_sample = df_sample.join(
    weekly_df,
    on=["LCLid", "year", "week_of_year"],
    how="left"
)

In [ ]:
# ============================================================
# 8. Agrégations saisonnières
# ============================================================

season_df = df_sample.groupBy(
    "LCLid",
    "season"
).agg(
    avg("energy_kwh").alias("season_avg_energy"),
    max("energy_kwh").alias("season_peak_energy"),
    stddev("energy_kwh").alias("season_std_energy")
)

In [ ]:
# ============================================================
# 9. Jointure des agrégations saisonnières
# ============================================================

df_sample = df_sample.join(
    season_df,
    on=["LCLid", "season"],
    how="left"
)

In [ ]:
# ============================================================
# 10. Vérification finale sur le test
# ============================================================

df_sample.select(
    "LCLid",
    "tstp",
    "energy_kwh",
    "lag_1h",
    "lag_24h",
    "lag_7d",
    "rolling_mean_24h",
    "rolling_std_24h",
    "weekly_avg_energy",
    "weekly_peak_energy",
    "season_avg_energy",
    "season_peak_energy",
    "target_energy_next_30min"
).show(10, False)

In [ ]:
# ============================================================
# 11. Vérification des valeurs nulles
# ============================================================

df_sample.select(
    sum(col("rolling_mean_24h").isNull().cast("int")).alias("null_rolling_mean_24h"),
    sum(col("rolling_std_24h").isNull().cast("int")).alias("null_rolling_std_24h"),
    sum(col("weekly_avg_energy").isNull().cast("int")).alias("null_weekly_avg"),
    sum(col("season_avg_energy").isNull().cast("int")).alias("null_season_avg"),
    sum(col("target_energy_next_30min").isNull().cast("int")).alias("null_target")
).show()

In [ ]:
# ============================================================
# 12. Sauvegarde du dataset test
# ============================================================

df_sample.write.format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset_test/"
    )

In [ ]:
# ============================================================
# 13. Exécution finale sur tout le dataset
# ============================================================

df_full = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/temporal_features/"
)

df_full = df_full.repartition("year", "month")

In [ ]:
# ============================================================
# 14. Rolling Features sur tout le dataset
# ============================================================

window_spec = Window.partitionBy("LCLid").orderBy("tstp")

rolling_24h = window_spec.rowsBetween(-48, 0)

df_full = df_full.withColumn(
    "rolling_mean_24h",
    avg("energy_kwh").over(rolling_24h)
).withColumn(
    "rolling_std_24h",
    stddev("energy_kwh").over(rolling_24h)
).withColumn(
    "rolling_max_24h",
    max("energy_kwh").over(rolling_24h)
).withColumn(
    "rolling_min_24h",
    min("energy_kwh").over(rolling_24h)
)

In [ ]:
# ============================================================
# 15. Agrégations hebdomadaires sur tout le dataset
# ============================================================

weekly_full = df_full.groupBy(
    "LCLid",
    "year",
    "week_of_year"
).agg(
    avg("energy_kwh").alias("weekly_avg_energy"),
    max("energy_kwh").alias("weekly_peak_energy"),
    stddev("energy_kwh").alias("weekly_std_energy")
)

df_full = df_full.join(
    weekly_full,
    on=["LCLid", "year", "week_of_year"],
    how="left"
)

In [ ]:
# ============================================================
# 16. Agrégations saisonnières sur tout le dataset
# ============================================================

season_full = df_full.groupBy(
    "LCLid",
    "season"
).agg(
    avg("energy_kwh").alias("season_avg_energy"),
    max("energy_kwh").alias("season_peak_energy"),
    stddev("energy_kwh").alias("season_std_energy")
)

df_full = df_full.join(
    season_full,
    on=["LCLid", "season"],
    how="left"
)

In [ ]:
# ============================================================
# 17. Sauvegarde finale du dataset ML Ready
# ============================================================

df_full.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(
        "abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset/"
    )

In [ ]:
# ============================================================
# 18. Vérification de la sauvegarde finale
# ============================================================

test_final = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset/"
)

test_final.select(
    "LCLid",
    "tstp",
    "energy_kwh",
    "rolling_mean_24h",
    "rolling_std_24h",
    "weekly_avg_energy",
    "season_avg_energy"
).show(5, False)